## Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  str    
 3   color       53943 non-null  str    
 4   clarity     53943 non-null  str    
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), str(3)
memory usage: 4.5 MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2

## Tiền xử lý dữ liệu

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


## Chia tập dữ liệu

In [3]:
from sklearn.model_selection import train_test_split

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu train test
X_train, X_test, y_train, y_test = train_test_split(
      X,
      y,
      test_size=0.2,
      random_state=42
    )

# In ra kết quả chia dữ liệu
print(f"Có {X_train.shape[0]} mẫu huấn luyện và {X_test.shape[0]} mẫu kiểm tra")

Có 43020 mẫu huấn luyện và 10755 mẫu kiểm tra


## Training

In [ ]:
# Import mô hình cây quyết định hồi quy
from sklearn.tree import DecisionTreeRegressor
# Import các hàm đánh giá mô hình: MAE, MSE, R²
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Import hàm tính căn bậc hai
from math import sqrt
# Import thư viện đo thời gian
import time

# Khai báo các giá trị max_depth cần thử (từ 1 đến 12)
max_depth_values = range(1, 13)
# Khai báo các giá trị min_samples_leaf cần thử (từ 1 đến 8)
min_samples_leaf_values = range(1, 9)
# Khai báo các giá trị min_samples_split cần thử (từ 2 đến 12)
min_samples_split_values = range(2, 13)

# Khởi tạo RMSE tốt nhất bằng vô cùng để mọi kết quả RMSE hợp lệ đều nhỏ hơn
best_rmse = float('inf')
# Khởi tạo biến lưu bộ tham số tốt nhất
best_params = None
# Khởi tạo biến lưu thời gian tốt nhất
best_times = None
# Khởi tạo biến lưu mô hình có kết quả tốt nhất
best_model = None

# Duyệt qua từng giá trị max_depth
for max_depth in max_depth_values:
    # Duyệt qua từng giá trị min_samples_leaf
    for min_samples_leaf in min_samples_leaf_values:
        # Duyệt qua từng giá trị min_samples_leaf
        for min_samples_split in min_samples_split_values:

            # Khởi tạo mô hình cây quyết định hồi quy với bộ tham số hiện tại
            model = DecisionTreeRegressor(
                criterion="squared_error",  # Chọn cách chia nút dựa trên việc giảm tổng sai số bình phương
                random_state=42,  # Cố định giá trị ngẫu nhiên để kết quả có thể tái lập
                max_depth=max_depth,  # Quy định độ sâu tối đa của cây
                min_samples_leaf=min_samples_leaf,  # Quy định số mẫu tối thiểu tại mỗi nút lá
                min_samples_split=min_samples_split # Quy định số nút lá tối thiểu
            )

            # Lấy thời điểm bắt đầu
            start_time = time.perf_counter()
            # Huấn luyện mô hình bằng tập dữ liệu Train
            model.fit(X_train, y_train)
            # Lấy thời điểm kết thúc
            end_time = time.perf_counter()

            # Dự đoán giá kim cương trên tập dữ liệu Test
            y_pred_test = model.predict(X_test)

            # Tính căn bậc hai của sai số bình phương trung bình (RMSE)
            rmse = sqrt(mean_squared_error(y_test, y_pred_test))

            # Kiểm tra xem RMSE hiện tại có tốt hơn kết quả tốt nhất trước đó không
            if rmse < best_rmse:
                # Cập nhật RMSE tốt nhất
                best_rmse = rmse
                # Lưu lại bộ tham số tạo ra RMSE tốt nhất
                best_params = {
                    'max_depth': max_depth,
                    'min_samples_leaf': min_samples_leaf,
                    'min_samples_split': min_samples_split
                }
                # Lưu lại thời gian tương ứng với bộ tham số tốt nhất
                best_times = end_time - start_time
                # Lưu lại mô hình tương ứng với bộ tham số tốt nhất
                best_model = model

# In ra bộ tham số tốt nhất tìm được
print("Tham số tốt nhất:", best_params)

# Dự đoán giá kim cương trên tập Train bằng mô hình tốt nhất
y_pred_train = best_model.predict(X_train)
# In MAE (Mean Absolute Error): Sai số tuyệt đối trung bình
print("\nMAE:", mean_absolute_error(y_train, y_pred_train))
# In MSE (Mean Squared Error): Sai số bình phương trung bình
print("MSE:", mean_squared_error(y_train, y_pred_train))
# In RMSE (Root Mean Squared Error): Căn bậc hai của MSE
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
# In R² (R-squared): Hệ số xác định
print("R^2:", r2_score(y_train, y_pred_train))

# Dự đoán giá kim cương trên tập Test bằng mô hình tốt nhất
y_pred_test = best_model.predict(X_test)
# In MAE (Mean Absolute Error): Sai số tuyệt đối trung bình
print("\nMAE:", mean_absolute_error(y_test, y_pred_test))
# In MSE (Mean Squared Error): Sai số bình phương trung bình
print("MSE:", mean_squared_error(y_test, y_pred_test))
# In RMSE (Root Mean Squared Error): Căn bậc hai của MSE
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
# In R² (R-squared): Hệ số xác định
print("R^2:", r2_score(y_test, y_pred_test))

# In ra thời gian tương ứng với bộ tham số tốt nhất
print(f"\nThời gian huấn luyện: {best_times:.4f}")

Tham số tốt nhất: {'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 11}

MAE: 251.1993134213399
MSE: 227348.29837599673
RMSE: 476.8105476769539
R^2: 0.9857087831121231

MAE: 309.0905055861483
MSE: 342670.6417205197
RMSE: 585.3807664422532
R^2: 0.9783164212292579

Thời gian tốt nhất trên tập Test: 0.1048
